In [32]:
%load_ext autoreload
%autoreload 2
import warnings
from pandas.errors import SettingWithCopyWarning

warnings.simplefilter(action="ignore", category=SettingWithCopyWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import json5,json
import fitz #type: ignore
import json
import csv
from typing import Union, Dict, Any, List

from app.logger import *
from app.amc.fund_data import *
from app.insur.fund_data import *
from app.utils import *
from app.konstant import get_config, get_regex

utils = Helper()

def json_to_csv(input_data: str,output_file: str,spacing: int = 2) -> int:
    """
    Convert mutual fund JSON into a structured CSV.
    Parameters:
        output_file (str): Path to output CSV file
        spacing (int): Number of empty rows after each mutual fund
    Returns:
        int: Total number of rows written (excluding header)
    Raises:
        ValueError: If input data format is invalid
        IOError: If file operations fail
    """
    try:
        with open(input_data, "r", encoding="utf-8") as f:
            data = json.load(f)
    except Exception:
        raise

    # Validate
    if "records" not in data or not isinstance(data["records"], list):
        raise ValueError("Invalid JSON structure")

    row_count = 0
    try:
        with open(output_file, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            # Header
            writer.writerow([
                "page",
                "table",
                "sfin",
                "mutual_fund_name",
                "main_scheme_name",
                "portfolio_data_0",
                "portfolio_data_1",
                "monthly_aum_value",
                "empty_col",
            ])

            # Process each record
            for record in data["records"]:
                value = record.get("value", {})
                mf_name = value.get("mutual_fund_name", "")
                scheme_name = value.get("main_scheme_name", "")
                aum_value = value.get("monthly_aaum_value", "")
                portfolio_list: List[Dict[str, Any]] = value.get("portfolio_data", [])
                sfin = value.get("sfin","")

                # Skip if no portfolio data
                if not isinstance(portfolio_list, list):
                    continue

                for item in portfolio_list:
                    writer.writerow([
                        item.get("page", ""),
                        item.get("table", ""),
                        sfin,
                        mf_name,
                        scheme_name,
                        item.get("0", ""),
                        item.get("1", ""),
                        aum_value,
                        "",
                    ])
                    row_count += 1

                # Add spacing rows
                for _ in range(spacing):
                    writer.writerow([])
        return row_count
    except Exception as e:
        raise IOError(f"Error writing CSV: {e}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [62]:
config_path = r"C:\Users\kaustubh.keny\Projects\OFFICE PROJECTS\rep_fsparse\config\2026"

config_code = [
    73, 74, 75, 76, 77, 78, 79, 80,
    81, 82, 83, 84, 85, 86, 87, 88,
    89, 90, 91, 92, 93, 94, 112
]


keys_to_delete = ["line_x", "line_side", "update_min_add", "stop_words","method","sanitize_fund"]

for file in os.listdir(config_path):

    code = int(file.split("_")[0])

    if code not in config_code:
        continue

    fp = os.path.join(config_path, file)

    data = utils.load_json5(fp)

    params = data["PARAMS"]

    for key in keys_to_delete:
        params.pop(key, None)   # ✅ safe delete

    utils.save_json5(data, fp)


ValueError: invalid literal for int() with base 10: 'parameters.json5'

In [ ]:
config_path = r"C:\Users\kaustubh.keny\Projects\OFFICE PROJECTS\rep_fsparse\config\2026"

config_code = [
    73, 74, 75, 76, 77, 78, 79, 80,
    81, 82, 83, 84, 85, 86, 87, 88,
    89, 90, 91, 92, 93, 94, 112
]

import os

utils = Helper()

for file in os.listdir(config_path):

    code = int(file.split("_")[0])  # ✅ FIX 1: convert to int

    if code not in config_code:
        continue   # ✅ FIX 2: don't break loop

    fp = os.path.join(config_path, file)

    data = utils.load_json5(fp)
    dt = data.copy()

    params = data["PARAMS"]["port_bbox"]  # this is a LIST

    new_params = []

    for block in params:   # ✅ block is each dict
        temp_anchor = {}

        for key, value in block.items():
            if not isinstance(value, list):
                temp_anchor[key] = value

        # ✅ Add anchor field
        block["anchor"] = temp_anchor

        new_params.append(block)

    dt["PARAMS"]["port_bbox"] = new_params

    utils.save_json5(dt, fp)

In [49]:
#INSURANCE FUND
amc_id = '91_0'
path = r"91_30-Apr-26_IF.pdf"
config = get_config("2026",amc_id)
regex = get_regex("2026")

object = SBILifeINSR(config,regex,path)
title,path_pdf= object.check_and_highlight(path)

data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)

C:\Users\kaustubh.keny\Projects\OFFICE PROJECTS\rep_fsparse\config\2026\91_0_AMC.json5


In [50]:
title

{4: 'EQUITY FUND',
 5: 'BOND FUND',
 6: 'GROWTH FUND',
 7: 'BALANCED FUND',
 8: 'EQUITY OPTIMISER FUND',
 9: 'INDEX FUND',
 10: 'TOP 300 FUND',
 11: 'PE MANAGED FUND',
 12: 'EQUITY ELITE FUND',
 13: 'EQUITY ELITE II FUND',
 14: 'EQUITY PENSION FUND',
 15: 'BOND PENSION FUND',
 16: 'GROWTH PENSION FUND',
 17: 'BALANCED PENSION FUND',
 18: 'EQUITY OPTIMISER PENSION FUND',
 19: 'INDEX PENSION FUND',
 20: 'TOP 300 PENSION FUND',
 21: 'MONEY MARKET FUND',
 22: 'MONEY MARKET PENSION FUND',
 23: 'GPF070211 GUARANTEED PENSION FUND',
 24: 'DISCONTINUED POLICY FUND',
 25: 'EQUITY PENSION FUND II',
 26: 'BOND PENSION FUND II',
 27: 'MONEY MARKET PENSION FUND II',
 28: 'DISCONTINUE PENSION FUND',
 29: 'PURE FUND',
 30: 'MIDCAP FUND',
 31: 'BOND OPTIMISER FUND',
 32: 'CORPORATE BOND FUND',
 33: 'BLUECHIP FUND'}

In [51]:
# object = GeneraliLifeINSR(config,regex,path)
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text)

In [52]:
with open("extract.json","w+") as file:
    json.dump(extracted_text,file)
  
with open("data.json","w+") as file:
    json.dump(final_text,file)
    
save_path = os.path.join(object.JSON_PATH, object.FILE_NAME).replace(".pdf", ".json")
with open(save_path, 'w') as f:
  json.dump(dfs, f, indent=2)
    
print(f"File Saved At: {save_path}")

File Saved At: C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\91_30-Apr-26_IF.json


In [ ]:
pattern = "Rs.\\s*([\\d,.]+)"
for fund, content in final_text.items():
    # check = 'before.fund_manager'
    for key in content:
        if key.endswith(".aum"):
            print(fund)
            text =re.sub("[^A-Za-z0-9\\s\\-\\(\\)\\.\\,\\+\\%\\:\\&]+", "",content[key]).strip()
            print(text)
            match = re.findall(pattern,text, re.IGNORECASE)
            print(match)

In [48]:
path = r"C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\81_30-Apr-26_IF.json"
# path = r"C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\78_30-Apr-26_IF.json" #AXA
# path = r"C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\74_30-Apr-26_IF.json" #Bandhan
path = r"C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\89_30-Apr-26_IF.json"
path = r"C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\88_30-Apr-26_IF.json"
path = r"C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\88_30-Apr-26_1_IF.json"
path = r"C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\92_30-Apr-26_IF.json"
path = r"C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\87_30-Apr-26_IF.json"
path = r"C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\91_30-Apr-26_IF.json"
from pathlib import Path


_path_ = Path(path)
output_path = _path_.name.replace(".json",".csv")
json_to_csv(path,output_path)

848